In [1]:
import pandas as pd
import ast
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.utils import embedding_functions
import numpy as np

c:\Users\falak\Desktop\Seryn_revamp\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ==============================
# 1. LOAD AND PREPARE DATASETS
# ==============================

# Load your datasets
cleaned_df = pd.read_csv("cleaned_dataset.csv")

# Since food_df should come from the previous cleaning code, let's recreate it
food_cols = ["Breakfast_Weekly", "Lunch_Weekly", "Dinner_Weekly", "Snacks_Weekly", "Drinks_Weekly"]

def clean_food_list(food_str):
    """Convert stringified list to clean list of foods."""
    try:
        foods = ast.literal_eval(food_str)
        if not isinstance(foods, list):
            return []
        foods = [f.strip().lower() for f in foods if f and f.strip()]
        return list(set(foods))
    except (ValueError, SyntaxError):
        return []

# Create food dictionary (long format)
food_dict_rows = []
for idx, row in cleaned_df.iterrows():
    patient_id = idx
    for meal in food_cols:
        food_list = clean_food_list(row[meal])
        for food in food_list:
            food_dict_rows.append({
                "PatientID": patient_id,
                "MealType": meal.replace("_Weekly", ""),
                "FoodItem": food
            })

food_df = pd.DataFrame(food_dict_rows)

print(f"✅ Loaded cleaned dataset: {cleaned_df.shape}")
print(f"✅ Created food dataset: {food_df.shape}")
print("\nFood Dataset Columns:", food_df.columns.tolist())
print("Cleaned Dataset Columns:", cleaned_df.columns.tolist())


In [2]:
# ==============================
# 2. INITIALIZE EMBEDDING MODEL & CHROMADB
# ==============================

# Choose a SentenceTransformer model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Initialize Chroma client (local persistence)
client = chromadb.PersistentClient(path="./chroma_db")

# Create embedding function wrapper
hf_embed = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
# ==============================
# 3. CREATE COLLECTIONS
# ==============================

# Delete existing collections if you want fresh start (optional)
try:
    client.delete_collection("food_items")
    client.delete_collection("patient_profiles")
    print("🗑️ Deleted existing collections")
except:
    pass

# Create collections
food_collection = client.get_or_create_collection(
    name="food_items", 
    embedding_function=hf_embed,
    metadata={"description": "Individual food items from patient diets"}
)

patient_collection = client.get_or_create_collection(
    name="patient_profiles", 
    embedding_function=hf_embed,
    metadata={"description": "Patient dietary profiles and metadata"}
)

In [ ]:

# ==============================
# 4. PROCESS FOOD EMBEDDINGS IN BATCHES
# ==============================

print("\n📊 Processing Food Embeddings...")

# Get unique food items to avoid duplicates
unique_foods = food_df["FoodItem"].unique()
print(f"Found {len(unique_foods)} unique food items")

# Batch processing for food items
BATCH_SIZE = 100
total_foods = len(unique_foods)

for i in range(0, total_foods, BATCH_SIZE):
    batch_end = min(i + BATCH_SIZE, total_foods)
    batch_foods = unique_foods[i:batch_end]
    
    # Convert to list and create embeddings
    batch_texts = batch_foods.tolist()
    batch_embeddings = model.encode(batch_texts, show_progress_bar=False)
    
    # Generate unique IDs
    batch_ids = [f"food_{j}" for j in range(i, batch_end)]
    
    # Add metadata for each food item
    batch_metadata = []
    for food in batch_foods:
        # Find which meal types this food appears in
        food_rows = food_df[food_df["FoodItem"] == food]
        meal_types = food_rows["MealType"].unique().tolist()
        patient_count = food_rows["PatientID"].nunique()
        
        batch_metadata.append({
            "food_name": food,
            "meal_types": ", ".join(meal_types),
            "patient_count": patient_count
        })
    
    # Add to ChromaDB
    food_collection.add(
        documents=batch_texts,
        embeddings=batch_embeddings.tolist(),
        ids=batch_ids,
        metadatas=batch_metadata
    )
    
    print(f"  Processed {batch_end}/{total_foods} food items")

print(f"✅ Added {total_foods} unique food items to ChromaDB")

In [ ]:
# ==============================
# 5. PROCESS PATIENT PROFILE EMBEDDINGS IN BATCHES
# ==============================

print("\n👥 Processing Patient Profile Embeddings...")

# Create comprehensive patient profiles for embedding
patient_profiles = []
patient_metadata = []

for idx, row in cleaned_df.iterrows():
    # Create a text representation of the patient profile
    # Combine all relevant features into a searchable text
    
    # Collect all foods for this patient
    patient_foods = []
    for meal in food_cols:
        foods = clean_food_list(row[meal])
        patient_foods.extend(foods)
    
    # Create profile text (this will be embedded)
    profile_text = f"""
    Patient Profile:
    Age: {row['Age']:.2f}, Sex: {row['Sex']}, BMI: {row['BMI']:.2f}
    Diabetes Type: {row['DiabetesType']}, HbA1c: {row['HbA1c']:.2f}
    Activity Level: {row['ActivityLevel']}, Calorie Needs: {row['CalorieNeeds']:.2f}
    Carb Tolerance: {row['CarbTolerance']:.2f}, Sugar Sensitivity: {row['SugarSensitivity']}
    Diet includes: {', '.join(patient_foods[:10])}
    """
    
    patient_profiles.append(profile_text.strip())
    
    # Store structured metadata
    patient_metadata.append({
        "patient_id": idx,
        "age": float(row['Age']),
        "sex": int(row['Sex']),
        "bmi": float(row['BMI']),
        "diabetes_type": int(row['DiabetesType']),
        "hba1c": float(row['HbA1c']),
        "activity_level": int(row['ActivityLevel']),
        "calorie_needs": float(row['CalorieNeeds']),
        "carb_tolerance": float(row['CarbTolerance']),
        "sugar_sensitivity": int(row['SugarSensitivity']),
        "total_foods": len(patient_foods)
    })

# Batch processing for patient profiles
total_patients = len(patient_profiles)

for i in range(0, total_patients, BATCH_SIZE):
    batch_end = min(i + BATCH_SIZE, total_patients)
    
    batch_profiles = patient_profiles[i:batch_end]
    batch_meta = patient_metadata[i:batch_end]
    batch_embeddings = model.encode(batch_profiles, show_progress_bar=False)
    batch_ids = [f"patient_{j}" for j in range(i, batch_end)]
    
    # Add to ChromaDB
    patient_collection.add(
        documents=batch_profiles,
        embeddings=batch_embeddings.tolist(),
        ids=batch_ids,
        metadatas=batch_meta
    )
    
    print(f"  Processed {batch_end}/{total_patients} patient profiles")

print(f"✅ Added {total_patients} patient profiles to ChromaDB")

# ==============================
# 6. CREATE MEAL-SPECIFIC COLLECTIONS (Optional)
# ==============================

print("\n🍽️ Creating Meal-Specific Collections...")

meal_types = ["Breakfast", "Lunch", "Dinner", "Snacks", "Drinks"]

for meal_type in meal_types:
    collection_name = f"{meal_type.lower()}_foods"
    
    # Create collection for this meal type
    meal_collection = client.get_or_create_collection(
        name=collection_name,
        embedding_function=hf_embed,
        metadata={"meal_type": meal_type}
    )
    
    # Filter foods for this meal type
    meal_foods = food_df[food_df["MealType"] == meal_type]["FoodItem"].unique()
    
    if len(meal_foods) > 0:
        # Process in batches
        for i in range(0, len(meal_foods), BATCH_SIZE):
            batch_end = min(i + BATCH_SIZE, len(meal_foods))
            batch = meal_foods[i:batch_end].tolist()
            
            batch_embeddings = model.encode(batch, show_progress_bar=False)
            batch_ids = [f"{meal_type.lower()}_{j}" for j in range(i, batch_end)]
            
            meal_collection.add(
                documents=batch,
                embeddings=batch_embeddings.tolist(),
                ids=batch_ids,
                metadatas=[{"meal_type": meal_type, "food": food} for food in batch]
            )
        
        print(f"  ✅ {meal_type}: {len(meal_foods)} foods")


In [5]:
# ==============================
# 7. VERIFY & TEST
# ==============================

print("\n🔍 Testing Semantic Search...")

# Test 1: Find similar foods
test_food = "hummus"
food_collection = client.get_collection("food_items")
results = food_collection.query(
    query_texts=[test_food],
    n_results=5
)
print(f"\nFoods similar to '{test_food}':")
for doc, dist in zip(results['documents'][0], results['distances'][0]):
    print(f"  - {doc} (distance: {dist:.3f})")

# Test 2: Find patients with similar dietary needs
test_profile = "High BMI diabetic patient needing low sugar options"
patient_collection = client.get_collection("patient_profiles")
results = patient_collection.query(
    query_texts=[test_profile],
    n_results=3
)
print(f"\nPatients matching profile: '{test_profile}'")
for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
    print(f"  Patient {meta['patient_id']}: BMI={meta['bmi']:.2f}, HbA1c={meta['hba1c']:.2f}")

# Test 3: Find breakfast alternatives
dinner_results = client.get_collection("dinner_foods").query(
    query_texts=["unhealthy high sugar dinner"],
    n_results=5
)
print(f"\nUnhealthy dinner options:")
for doc in dinner_results['documents'][0]:
    print(f"  - {doc}")



🔍 Testing Semantic Search...

Foods similar to 'hummus':
  - sweet potato hummus (distance: 0.186)
  - stuffed peppers (veggie) (distance: 0.527)
  - boiled chickpeas (distance: 0.533)
  - chickpea salad (distance: 0.537)
  - boiled corn (distance: 0.575)

Patients matching profile: 'High BMI diabetic patient needing low sugar options'
  Patient 19885: BMI=1.27, HbA1c=-1.66
  Patient 18697: BMI=-0.71, HbA1c=-0.98
  Patient 70698: BMI=-0.85, HbA1c=-1.21

Unhealthy dinner options:
  - grilled halloumi salad
  - mushroom stew
  - mixed bean stew
  - vegetable stew
  - barley soup


In [ ]:
# ==============================
# 8. SUMMARY STATISTICS
# ==============================

print("\n📈 ChromaDB Summary:")
print(f"  - Food items collection: {food_collection.count()} items")
print(f"  - Patient profiles collection: {patient_collection.count()} profiles")
for meal_type in meal_types:
    collection = client.get_collection(f"{meal_type.lower()}_foods")
    print(f"  - {meal_type} foods: {collection.count()} items")

print("\n✅ All embeddings created and stored successfully!")
print(f"📁 Database saved to: ./chroma_db")


In [ ]:
# ==============================
# 9. UTILITY FUNCTIONS FOR FUTURE USE
# ==============================

def find_food_substitutes(food_item, n=5, meal_type=None):
    """Find substitute foods similar to the given item."""
    if meal_type:
        collection = client.get_collection(f"{meal_type.lower()}_foods")
    else:
        collection = food_collection
    
    results = collection.query(
        query_texts=[food_item],
        n_results=n+1  # +1 because the item itself might be in results
    )
    
    # Filter out the exact match if present
    substitutes = [doc for doc in results['documents'][0] if doc.lower() != food_item.lower()]
    return substitutes[:n]

def find_similar_patients(patient_profile, n=5):
    """Find patients with similar dietary needs."""
    results = patient_collection.query(
        query_texts=[patient_profile],
        n_results=n
    )
    return results

def get_meal_recommendations(patient_metadata, meal_type, n=10):
    """Get meal recommendations based on patient profile."""
    # Create a query based on patient needs
    query = f"""
    Suitable {meal_type} for patient with:
    BMI: {patient_metadata.get('bmi', 'normal')}
    Diabetes Type: {patient_metadata.get('diabetes_type', 'Type 2')}
    Sugar Sensitivity: {patient_metadata.get('sugar_sensitivity', 'moderate')}
    Looking for healthy, diabetic-friendly options
    """
    
    collection = client.get_collection(f"{meal_type.lower()}_foods")
    results = collection.query(
        query_texts=[query],
        n_results=n
    )
    return results['documents'][0]

print("\n🎯 Utility functions created:")
print("  - find_food_substitutes(food_item, n=5, meal_type=None)")
print("  - find_similar_patients(patient_profile, n=5)")
print("  - get_meal_recommendations(patient_metadata, meal_type, n=10)")